# Assignment 1

Computer Vision — COMP9320

In [ ]:
# Import the libraries used throughout the assignment.
%matplotlib inline
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Set the input and output paths relative to the project or notebook directory.
ASSIGNMENT_DIR = Path.cwd()
if not (ASSIGNMENT_DIR / "data" / "uni.jpeg").is_file():
    ASSIGNMENT_DIR = ASSIGNMENT_DIR / "Assignments" / "Assign_1"
DATA_DIR = ASSIGNMENT_DIR / "data"
OUTPUT_DIR = ASSIGNMENT_DIR / "outputs"

## Task 1: Image Loading and Grayscale Conversion

### 1. Use OpenCV to read the image uni.jpeg.

In [ ]:
# Read the original color image in OpenCV BGR format.
image_path = DATA_DIR / "uni.jpeg"
original_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
if original_bgr is None:
    raise FileNotFoundError(f"Unable to read image: {image_path}")

### 2. Display the original image and report its properties.

#### Spatial dimensions

- **Pixel:** one spatial sample of a digital image; a color pixel contains values for its color channels.
- **Spatial dimensions:** image height (rows) and width (columns), measured in pixels. For a color array, `shape` is `(height, width, channels)`; `shape[:2]` selects only the spatial dimensions.
- **BGR and RGB:** channel orders for blue, green, and red values. OpenCV loads this image in BGR order; Matplotlib expects RGB. Reordering channels preserves the spatial dimensions and displays the intended colors.

In [ ]:
# Display the original image in RGB order and report its height and width.
original_rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
height, width = original_bgr.shape[:2]
print(f"Spatial dimensions (height x width): {height} x {width} pixels")

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(original_rgb)
ax.set_title("Original image — uni.jpeg")
ax.set_xlabel("Column (pixels)")
ax.set_ylabel("Row (pixels)")
plt.tight_layout()
plt.show()

**Observation:** The image has 476 rows and 645 columns, so it is wider than it is tall. Array indexing uses `(row, column)`; the plot uses columns horizontally and rows vertically.

**References — documentation and further reading:**
- [OpenCV: Basic Operations on Images](https://docs.opencv.org/4.x/d3/df2/tutorial_py_basic_ops.html) — pixel values, BGR order, and array shape.
- [Matplotlib: imshow](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html) — RGB display and image coordinates.

#### Number of color channels

A **color channel** is a two-dimensional array containing one color component at every pixel. This image has blue, green, and red channels. OpenCV stores them in BGR order; each pixel therefore contains three values. The third element of `shape` gives the channel count, whereas `ndim` counts array axes.

In [ ]:
# Report the color channel count, channel order, and array structure.
channel_count = original_bgr.shape[2]
print(f"Number of color channels: {channel_count}")
print("OpenCV channel order: BGR (Blue, Green, Red)")
print("Display channel order: RGB (Red, Green, Blue)")
print(f"Image array shape (height, width, channels): {original_bgr.shape}")
print(f"Number of array axes: {original_bgr.ndim}")
for index, name in enumerate(("Blue", "Green", "Red")):
    print(f"Channel {index}: {name}; spatial shape: {original_bgr[:, :, index].shape}")

**Observation:** There are three color channels, each with 476 rows and 645 columns. The channel axis stores color components; it is not an additional spatial dimension.

**Reference — documentation:** [OpenCV: Basic Operations on Images](https://docs.opencv.org/4.x/d3/df2/tutorial_py_basic_ops.html), sections on image properties and splitting channels.

#### Data type and numerical intensity range

- **Data type (`dtype`):** the representation used to store each channel value. `uint8` is an unsigned 8-bit integer: it represents 256 integer levels from 0 to 255.
- **Channel intensity:** the stored value of one color component at a pixel. Zero is the minimum component level; 255 is the maximum. A single channel value is not the overall brightness of a color pixel.
- **Representable range:** the limits allowed by the data type, obtained with `np.iinfo`. **Observed range:** the minimum and maximum actually present in the image, obtained with `min()` and `max()`. These ranges need not be equal.

In [ ]:
# Report the data type, representable limits, and observed channel intensities.
dtype_limits = np.iinfo(original_bgr.dtype)
print(f"Data type: {original_bgr.dtype}")
print(f"Bits per channel value: {dtype_limits.bits}")
print(f"Representable intensity range: [{dtype_limits.min}, {dtype_limits.max}]")
print(f"Representable intensity levels: {int(dtype_limits.max) - int(dtype_limits.min) + 1}")
print(f"Observed intensity range across all channels: [{original_bgr.min()}, {original_bgr.max()}]")
for index, name in enumerate(("Blue", "Green", "Red")):
    channel = original_bgr[:, :, index]
    print(f"Observed {name} channel range: [{channel.min()}, {channel.max()}]")

**Observation:** The image uses `uint8`. The observed range is [0, 255] both overall and in each color channel. Each channel contains at least one minimum and one maximum value; this does not establish that all 256 levels occur or that a completely black or white pixel exists.

**Reference — documentation:** [NumPy: numpy.iinfo](https://numpy.org/doc/stable/reference/generated/numpy.iinfo.html) — integer limits and bit depth.

#### Memory requirements

- **Byte:** 8 bits. A `uint8` channel value occupies one byte (`itemsize`).
- **Array data memory:** the bytes occupied by the decoded pixel values (`nbytes`), excluding Python object metadata and other arrays. For this image: $M = H \times W \times C \times \text{bytes per channel value}$.
- **Units:** 1 KiB = 1,024 bytes; 1 MiB = 1,048,576 bytes; 1 MB = 1,000,000 bytes.
- **JPEG file size:** compressed storage on disk; it differs from the decoded array memory.

In [ ]:
# Calculate decoded image memory and compare it with the compressed JPEG file size.
pixel_count = height * width
bytes_per_value = original_bgr.itemsize
expected_bytes = pixel_count * channel_count * bytes_per_value
image_bytes = original_bgr.nbytes
jpeg_bytes = image_path.stat().st_size
print(f"Number of spatial pixels: {pixel_count:,}")
print(f"Number of stored channel values: {original_bgr.size:,}")
print(f"Bytes per channel value: {bytes_per_value}")
print(f"Bytes per color pixel: {channel_count * bytes_per_value}")
print(f"Memory calculation: {height} x {width} x {channel_count} x {bytes_per_value} = {expected_bytes:,} bytes")
print(f"Decoded array data (nbytes): {image_bytes:,} bytes")
print(f"Decoded array data: {image_bytes / 1024:.2f} KiB; {image_bytes / 1024**2:.4f} MiB; {image_bytes / 10**6:.5f} MB")
print(f"Compressed JPEG file: {jpeg_bytes:,} bytes ({jpeg_bytes / 1024:.2f} KiB)")
print(f"Calculated memory matches nbytes: {expected_bytes == image_bytes}")

**Observation:** The decoded BGR pixel data occupies 921,060 bytes, whereas the JPEG file occupies 67,589 bytes. JPEG stores a compressed representation. The reported array memory excludes the separate RGB display copy, plotting buffers, and Python object overhead.

**References — documentation:** [NumPy: ndarray.nbytes](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.nbytes.html) and [ndarray.itemsize](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.itemsize.html).

### 3. Convert the original image to grayscale.

**Grayscale** represents each pixel with one intensity value instead of three color components. For `uint8`, 0 represents black and 255 white. OpenCV combines the channels using:

$$Y = 0.299R + 0.587G + 0.114B.$$

The unequal weights approximate the different contributions of the color components to perceived brightness; this is not a simple channel average or a physical luminance measurement. `COLOR_BGR2GRAY` matches the input BGR order. Integer conversion quantizes the result, so manually computed floating-point values may differ slightly.

The conversion preserves height and width but discards color information; the original colors cannot generally be recovered from the grayscale image.

In [ ]:
# Convert the original BGR image to one grayscale channel and report its properties.
gray = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2GRAY)
print("Conversion: cv2.COLOR_BGR2GRAY")
print("Weighted intensity: Y = 0.299R + 0.587G + 0.114B")
print(f"Grayscale shape (height, width): {gray.shape}")
print("Number of grayscale channels: 1")
print(f"Number of array axes: {gray.ndim}")
print(f"Data type: {gray.dtype}")
print(f"Observed intensity range: [{gray.min()}, {gray.max()}]")
print(f"Spatial dimensions preserved: {gray.shape == original_bgr.shape[:2]}")
print(f"Grayscale array data: {gray.nbytes:,} bytes")

**Observation:** The grayscale image is a two-dimensional `uint8` array of shape (476, 645), with intensities from 0 to 255. It stores one value per pixel and uses 307,020 bytes of pixel data, one-third of the original BGR array.

**Reference — documentation:** [OpenCV: Color conversions, RGB ↔ GRAY](https://docs.opencv.org/4.x/de/d25/imgproc_color_conversions.html) — conversion formula and channel-order flags.

### 4. Display the grayscale image.

A **colormap** maps stored values to display colors. `cmap="gray"` displays low intensities as dark and high intensities as light. Setting `vmin=0` and `vmax=255` fixes the display scale to the full 8-bit range; it does not modify the image values. `interpolation="nearest"` avoids smoothing between pixels during rendering.

In [ ]:
# Display the grayscale image with a fixed intensity scale and labeled axes.
fig, ax = plt.subplots(figsize=(8, 6))
gray_plot = ax.imshow(gray, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
ax.set_title("Grayscale image — uni.jpeg")
ax.set_xlabel("Column (pixels)")
ax.set_ylabel("Row (pixels)")
fig.colorbar(gray_plot, ax=ax, label="Grayscale intensity (0–255)")
plt.tight_layout()
plt.show()

**Observation:** The grayscale view retains the spatial layout while removing color distinctions. Dark and light areas represent low and high weighted intensities. Different colors can map to similar gray values, so some color-based distinctions may be lost.

**Reference — documentation:** [Matplotlib: imshow](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html) — colormaps, intensity limits, and interpolation.

## Task 2: Edge Detection Using Canny

### 1. Use the grayscale image obtained in Task 1.

An **edge** is a location of a rapid spatial change in image intensity. It may indicate an object boundary, texture, or a lighting change; not every detected edge is an object boundary. **Edge detection** identifies these changes to emphasize image structure.

Reuse `gray` from Task 1 as the single-channel input. No image reload or second grayscale conversion is needed.

In [ ]:
# Reuse the Task 1 grayscale image and validate the input for edge detection.
assert gray.ndim == 2, "Expected a single-channel grayscale image."
assert gray.dtype == np.uint8, "Expected an 8-bit unsigned integer image."
assert gray.shape == original_bgr.shape[:2], "Spatial dimensions do not match."
canny_input = gray
print("Input source: Task 1 grayscale image (gray)")
print(f"Input shape (height, width): {canny_input.shape}")
print("Number of channels: 1")
print(f"Input data type: {canny_input.dtype}")
print(f"Observed intensity range: [{canny_input.min()}, {canny_input.max()}]")
print(f"Same array as Task 1 grayscale image: {canny_input is gray}")

**Observation:** The input is the existing (476, 645) `uint8` grayscale array. `canny_input = gray` creates another reference to the same array, not a copy; this step does not change its values.

**Reference — further reading:** [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — intensity changes and the Canny pipeline.

### 2. Apply the Canny edge detection algorithm to extract the edges.
,
,**Gaussian smoothing** averages nearby pixels with greater weight near the center to reduce noise before detecting intensity changes. We explicitly apply a 5 × 5 Gaussian kernel with standard deviation $\sigma=1$ pixel.
,
,A **gradient** measures the spatial change in intensity. Canny uses gradients, thins candidate edges, then links them using **hysteresis**: strong responses above the upper threshold seed edges; weaker candidates between the thresholds survive only when connected to strong edges. Thresholds apply to gradient magnitude, not directly to grayscale values.
,
,Use initial thresholds of 50 and 150. `apertureSize=3` selects a 3 × 3 Sobel derivative kernel; `L2gradient=True` uses $\sqrt{G_x^2+G_y^2}$. The threshold choice will be assessed in Steps 5–6. A **binary edge map** contains two labels: 255 for detected edges and 0 for non-edges.

**Parameter glossary:**

| Term / variable | Meaning in this implementation |
| --- | --- |
| Kernel | A small matrix of filter weights applied over a pixel neighborhood. |
| Sobel operator | A discrete derivative filter approximating intensity changes along columns ($G_x$) and rows ($G_y$). It combines differentiation with smoothing in the perpendicular direction. |
| `sobel_aperture` | Our variable for the Sobel kernel side length; passed to Canny as `apertureSize`. A value of 3 means a 3 × 3 neighborhood, not three passes or a camera aperture. |
| `gaussian_kernel` | Width and height of the separate Gaussian smoothing kernel: (5, 5). |
| `gaussian_sigma` | Gaussian standard deviation in pixels; controls the spread of smoothing weights. |
| `use_l2_gradient` | Our Boolean variable passed as `L2gradient`; `True` selects Euclidean gradient magnitude. |
| `BORDER_REFLECT_101` | Extends image values by reflection without repeating the boundary pixel, allowing filtering near image borders. |

**Reference — documentation:** [OpenCV: Sobel Derivatives](https://docs.opencv.org/4.x/d2/d2c/tutorial_sobel_derivatives.html) — derivative kernels and gradient estimation. The Sobel kernel and the separate Gaussian kernel have different roles.

In [ ]:
# Smooth the grayscale input, apply Canny, and report the settings and output.
gaussian_kernel = (5, 5)
gaussian_sigma = 1.0
lower_threshold = 50
upper_threshold = 150
sobel_aperture = 3
use_l2_gradient = True
smoothed_gray = cv2.GaussianBlur(
    canny_input, gaussian_kernel, sigmaX=gaussian_sigma,
    sigmaY=gaussian_sigma, borderType=cv2.BORDER_REFLECT_101
)
edges = cv2.Canny(
    smoothed_gray, lower_threshold, upper_threshold,
    apertureSize=sobel_aperture, L2gradient=use_l2_gradient
)
edge_pixel_count = np.count_nonzero(edges)
print(f"Gaussian kernel: {gaussian_kernel}; sigma: {gaussian_sigma} pixel")
print("Gaussian border handling: BORDER_REFLECT_101")
print(f"Hysteresis thresholds (lower, upper): ({lower_threshold}, {upper_threshold})")
print(f"Sobel aperture size: {sobel_aperture}; L2gradient: {use_l2_gradient}")
print(f"Edge map shape: {edges.shape}; data type: {edges.dtype}")
print(f"Unique output values: {np.unique(edges).tolist()}")
print(f"Detected edge pixels: {edge_pixel_count:,} / {edges.size:,}")
print(f"Edge pixel percentage: {100 * edge_pixel_count / edges.size:.2f}%")

**Interpretation:** White pixels label retained edges; black pixels label non-edges. The edge pixel percentage describes output density, not detection accuracy. The same smoothing and gradient settings must be retained when comparing thresholds.

**References — documentation and further reading:** [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) and [Canny API](https://docs.opencv.org/4.x/dd/d1a/group__imgproc__feature.html) — algorithm and parameters.

### 3. Display the grayscale image and the resulting edge image side by side.

The **grayscale image** stores intensity values, whereas the **edge map** stores binary edge labels. Both panels use the same spatial coordinates and a fixed 0–255 display range. The left panel shows the original Task 1 grayscale image; Canny was applied to its Gaussian-smoothed version in Step 2.

In [ ]:
# Compare the original grayscale image with the Canny edge map at matching scales.
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
axes[0].imshow(canny_input, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
axes[0].set_title("Original grayscale image")
axes[1].imshow(edges, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
axes[1].set_title(f"Canny edges — thresholds {lower_threshold}, {upper_threshold}")
for ax in axes:
    ax.set_xlabel("Column (pixels)")
    ax.set_ylabel("Row (pixels)")
fig.suptitle("Grayscale and Canny edge comparison")
plt.tight_layout()
plt.show()

**Observation:** The edge map highlights building outlines, paths, and fine vegetation or architectural texture while discarding the grayscale shading. Some visible responses arise from texture and shadows rather than object boundaries. White pixels represent detected edges, not brighter scene regions; visual comparison alone does not establish detection accuracy.

**Reference — further reading:** [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — grayscale and edge-map comparison.

### 4. Explain the five fundamental stages of the Canny algorithm.

#### 4.1 Gaussian smoothing

**Smoothing** reduces rapid local intensity variations. A Gaussian filter replaces each pixel with a weighted average of its neighborhood, assigning larger weights near the center:

$$K(i,j)=\frac{\exp\left[-(i^2+j^2)/(2\sigma^2)\right]}{\sum_{u,v}\exp\left[-(u^2+v^2)/(2\sigma^2)\right]},\qquad I_s(x,y)=\sum_{i,j}K(i,j)I(x-i,y-j).$$

Here, $I$ is the grayscale image, $I_s$ is the smoothed image, and $(i,j)$ are kernel offsets. **Normalization** makes the finite kernel weights sum to 1, preserving constant intensities away from numerical rounding.

**Why before differentiation?** Derivatives respond strongly to rapid variations, including noise. Smoothing suppresses some unwanted responses, but also softens real edges and fine texture. Larger $\sigma$ generally increases this tradeoff when the kernel support is sufficient.

**Connection to Step 2:** `cv2.GaussianBlur` explicitly produces `smoothed_gray` with a 5 × 5 kernel and $\sigma_x=\sigma_y=1$. The separate Gaussian operation is not an additional Gaussian stage automatically applied by `cv2.Canny`. Sobel's built-in perpendicular smoothing does not replace this explicit preprocessing.

**References — documentation:** [OpenCV: Smoothing Images](https://docs.opencv.org/4.x/d4/d13/tutorial_py_filtering.html) — Gaussian kernels and sigma; [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — smoothing before gradient computation.

#### 4.2 Gradient computation

A **partial derivative** measures change along one coordinate while holding the other fixed. The image **gradient** is the vector $\nabla I_s=(G_x,G_y)$: $G_x$ approximates change along columns (left to right), and $G_y$ along rows (top to bottom). Sobel estimates these derivatives from neighboring pixels of the smoothed image.

For the 3 × 3 Sobel operator, the correlation kernels are:

$$K_x=\begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix},\qquad K_y=\begin{bmatrix}-1&-2&-1\\0&0&0\\1&2&1\end{bmatrix}.$$

**Correlation** multiplies each kernel weight by the corresponding neighborhood value and sums the products without flipping the kernel. Thus $G_x$ is positive for intensity increasing to the right, while $G_y$ is positive for intensity increasing downward. A large absolute response indicates a strong change; $G_x$ emphasizes vertical boundaries and $G_y$ horizontal boundaries.

**Code parameters:** `dx` and `dy` are derivative orders: `(1, 0)` computes the first x derivative, and `(0, 1)` the first y derivative. `ksize=sobel_aperture` selects the 3 × 3 kernel. `ddepth=cv2.CV_32F` stores signed 32-bit floating-point responses, preserving negative derivatives that `uint8` cannot represent. `BORDER_REPLICATE` extends the nearest boundary value for derivative filtering.

The following cell illustrates this stage separately; it does not replace the existing `cv2.Canny` result or claim to expose its internal buffers.

In [ ]:
# Compute signed Sobel derivatives of the Gaussian-smoothed image for illustration.
gradient_x = cv2.Sobel(
    smoothed_gray, cv2.CV_32F, dx=1, dy=0,
    ksize=sobel_aperture, borderType=cv2.BORDER_REPLICATE
)
gradient_y = cv2.Sobel(
    smoothed_gray, cv2.CV_32F, dx=0, dy=1,
    ksize=sobel_aperture, borderType=cv2.BORDER_REPLICATE
)
print(f"Sobel kernel size: {sobel_aperture} x {sobel_aperture}")
print("Derivative border handling: BORDER_REPLICATE")
for name, derivative in (("Gx: column-direction change", gradient_x),
                         ("Gy: row-direction change", gradient_y)):
    print(f"{name}; shape: {derivative.shape}; dtype: {derivative.dtype}")
    print(f"  Signed response range: [{derivative.min():.1f}, {derivative.max():.1f}]")

**Interpretation:** Derivative responses are signed changes, not grayscale intensities; their range is not restricted to 0–255. Negative responses indicate decreasing intensity in the corresponding coordinate direction. The next stage combines these components into magnitude and orientation.

**Reference — documentation:** [OpenCV: Sobel Derivatives](https://docs.opencv.org/4.x/d2/d2c/tutorial_sobel_derivatives.html) — derivative kernels, gradient components, and output depth.

#### 4.3 Gradient magnitude and orientation estimation

**Magnitude** is the length of the gradient vector and measures the strength of local intensity change. **Orientation** is the direction of greatest increase in intensity, perpendicular to the local edge tangent:

$$M=\sqrt{G_x^2+G_y^2},\qquad \theta=\operatorname{atan2}(G_y,G_x).$$

`np.hypot(Gx, Gy)` computes the Euclidean magnitude element by element. `np.arctan2(Gy, Gx)` uses both signs to select the correct quadrant and avoids explicit division by zero. Its argument order is y first, x second. It returns **radians** (a full turn is $2\pi$); `np.degrees` converts to **degrees** (a full turn is 360°).

Since image rows increase downward, 0° points right, +90° down, and −90° up. Where magnitude is zero, orientation is undefined; the code marks it as `NaN` (not a number). **Modulo 180°** maps opposite gradient directions to the same unoriented axis, useful for comparing neighbors during non-maximum suppression. These illustrative arrays do not expose Canny's internal buffers.

In [ ]:
# Calculate gradient strength and direction while excluding zero-gradient orientations.
gradient_magnitude = np.hypot(gradient_x, gradient_y)
orientation_degrees = np.degrees(np.arctan2(gradient_y, gradient_x))
valid_orientation = gradient_magnitude > 0
orientation_degrees = np.where(valid_orientation, orientation_degrees, np.nan)
orientation_axis = np.mod(orientation_degrees, 180.0)
print(f"Magnitude shape: {gradient_magnitude.shape}; dtype: {gradient_magnitude.dtype}")
print(f"Magnitude range: [{gradient_magnitude.min():.2f}, {gradient_magnitude.max():.2f}]")
print("Directed angle convention: degrees from +x; positive toward increasing rows")
print("Directed angle interval: [-180, 180] degrees")
print("Unoriented gradient axis interval: [0, 180) degrees")
print(f"Pixels with defined orientation: {np.count_nonzero(valid_orientation):,}")
print(f"Zero-gradient pixels (undefined orientation): {np.count_nonzero(~valid_orientation):,}")
if np.any(valid_orientation):
    print(f"Observed directed angle range: [{np.nanmin(orientation_degrees):.2f}, {np.nanmax(orientation_degrees):.2f}] degrees")

**Interpretation:** Magnitude is nonnegative and can exceed 255 because it is derived from signed Sobel responses, not stored grayscale intensity. A large magnitude alone does not establish an edge: non-maximum suppression and hysteresis still determine which responses survive. At very small magnitudes, orientation is sensitive to small intensity changes.

**References — documentation:** [NumPy: hypot](https://numpy.org/doc/stable/reference/generated/numpy.hypot.html), [NumPy: arctan2](https://numpy.org/doc/stable/reference/generated/numpy.arctan2.html), and [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html).

#### 4.4 Non-maximum suppression (NMS)

**Suppression** means setting a response to zero. A **local maximum** is a response at least as large as its nearby comparison values. NMS thins broad gradient ridges by retaining local peaks **along the gradient direction**, across the edge rather than along its tangent.

At pixel $p$, compare $M(p)$ with magnitudes on both sides along the gradient axis. Suppress it if either comparison is larger. A common discrete approximation groups orientations into 0°, 45°, 90°, and 135°:

| Gradient axis (image coordinates) | Neighbors compared |
| --- | --- |
| 0° | Left and right |
| 45° | Upper-left and lower-right |
| 90° | Above and below |
| 135° | Upper-right and lower-left |

**Tie handling** determines which equal neighboring peaks survive; implementations may use asymmetric comparisons. NMS retains selected magnitudes and zeros others; its conceptual output is not yet the final binary edge map. Hysteresis subsequently selects and links candidates.

The following one-dimensional profile illustrates a cross-section along the gradient direction. It uses strict comparisons for a unique peak and sets its two endpoints to zero; it is not a full-image NMS implementation or Canny's internal output.

In [ ]:
# Illustrate thinning of a gradient ridge using a unique local maximum.
nms_profile = np.array([0, 20, 60, 100, 70, 30, 0], dtype=np.float32)
nms_profile_thinned = np.zeros_like(nms_profile)
local_peak_mask = (
    (nms_profile[1:-1] > nms_profile[:-2])
    & (nms_profile[1:-1] > nms_profile[2:])
)
nms_profile_thinned[1:-1] = np.where(local_peak_mask, nms_profile[1:-1], 0)
print("Illustrative gradient magnitudes:", nms_profile.tolist())
print("After non-maximum suppression:", nms_profile_thinned.tolist())
print(f"Nonzero responses before: {np.count_nonzero(nms_profile)}")
print(f"Nonzero responses after: {np.count_nonzero(nms_profile_thinned)}")

**Interpretation:** The response 100 survives because it exceeds both neighbors. The other positive responses are suppressed, reducing a five-sample ridge to one sample. No lower or upper threshold is applied in this example; NMS compares neighboring strengths rather than selecting edges by absolute strength.

**Reference — documentation:** [OpenCV: Canny Edge Detection, Non-maximum Suppression](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — comparisons along the gradient direction.

#### 4.5 Hysteresis thresholding

**Hysteresis** uses two thresholds and connectivity to decide which thinned gradient responses form final edges. With lower threshold $T_L$ and upper threshold $T_H$ ($T_L<T_H$), this example uses the following explicit convention:

- **Strong edge seed:** a retained magnitude greater than $T_H$.
- **Weak candidate:** a retained magnitude greater than $T_L$ but no greater than $T_H$.
- **Rejected response:** a magnitude no greater than $T_L$.

**8-connectivity** allows a pixel to connect to its horizontal, vertical, and diagonal neighbors. Keep a weak candidate only if a path of eligible candidates connects it to a strong seed. The path may contain several weak pixels: direct adjacency to a strong seed is not required. An isolated weak component is discarded.

The thresholds apply to magnitudes after non-maximum suppression, not to the original grayscale intensities. The final **binary edge map** uses 255 for retained pixels and 0 otherwise.

The illustrative matrix below represents responses *after* NMS. `connectedComponents` assigns a distinct integer **label** to each connected foreground region; label 0 is background. Keeping components containing strong seeds illustrates hysteresis without reproducing Canny's internal implementation.

In [ ]:
# Illustrate hysteresis by keeping candidate components connected to strong seeds.
hysteresis_demo = np.array([
    [0, 200, 90, 70, 0, 0, 0],
    [0,   0,  0,  0, 0, 0, 0],
    [0,   0,  0,  0, 0, 0, 0],
    [0,  80, 60,  0, 0, 0, 0],
    [0,   0,  0,  0, 0, 0, 0]
], dtype=np.float32)
demo_low, demo_high = 50, 150
candidate_mask = hysteresis_demo > demo_low
strong_mask = hysteresis_demo > demo_high
component_count, component_labels = cv2.connectedComponents(
    candidate_mask.astype(np.uint8), connectivity=8
)
strong_component_labels = np.unique(component_labels[strong_mask])
keep_mask = candidate_mask & np.isin(component_labels, strong_component_labels)
hysteresis_result = (keep_mask.astype(np.uint8) * 255)
print(f"Illustrative thresholds: lower={demo_low}, upper={demo_high}")
print("Post-NMS magnitude example:\n", hysteresis_demo)
print("Final binary example:\n", hysteresis_result)
print(f"Candidate components (excluding background): {component_count - 1}")
print(f"Strong seeds: {np.count_nonzero(strong_mask)}")
print(f"Weak candidates retained: {np.count_nonzero(keep_mask & ~strong_mask)}")
print(f"Weak candidates discarded: {np.count_nonzero(candidate_mask & ~strong_mask & ~keep_mask)}")

**Interpretation:** The seed 200 retains the connected chain 90 → 70, even though 70 does not directly touch the seed. The disconnected weak responses 80 and 60 are discarded. Thus hysteresis can preserve weaker continuations while rejecting isolated weak responses. Connected noise or texture can still survive; connectivity does not guarantee an object boundary.

**Connection to Step 2:** `lower_threshold=50` and `upper_threshold=150` supply the two thresholds to `cv2.Canny`. The five stages are now explained; the following steps assess the selected thresholds and their effects.

**Reference — documentation:** [OpenCV: Canny Edge Detection, Hysteresis Thresholding](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — strong seeds and connected weak candidates.

### 5. Report suitable lower and upper hysteresis thresholds and justify their selection.

**Selected thresholds:** $T_L=50$ and $T_H=150$, with the existing 5 × 5 Gaussian smoothing ($\sigma=1$), 3 × 3 Sobel aperture, and L2 magnitude.

**Selection criterion:** retain recognizable building and path contours while limiting dense fine-texture responses. Visual inspection of the selected edge map shows a useful compromise: major outlines remain visible, although vegetation, windows, and shadows still produce responses. A lower pair (25, 75) retains noticeably denser texture; a higher pair (100, 200) suppresses more detail and weak contour segments. These are qualitative comparisons, not measured error rates.

The upper threshold selects strong seeds; the lower threshold admits weaker connected continuations. Their 3:1 ratio is a practical choice for this image and preprocessing, not a universal rule. Thresholds are on the Sobel gradient scale, so 150 is not a grayscale brightness cutoff. Changing smoothing, derivative aperture, or magnitude definition requires reassessing them.

No annotated reference edge map is available; therefore, the choice is visually suitable rather than proven optimal. Edge density alone cannot establish accuracy.

In [ ]:
# Report the selected thresholds and the fixed preprocessing used to justify them.
print(f"Selected lower threshold: {lower_threshold}")
print(f"Selected upper threshold: {upper_threshold}")
print(f"Upper-to-lower threshold ratio: {upper_threshold / lower_threshold:.1f}:1")
print(f"Gaussian kernel: {gaussian_kernel}; sigma: {gaussian_sigma}")
print(f"Sobel aperture: {sobel_aperture}; L2gradient: {use_l2_gradient}")
print(f"Selected edge pixels: {np.count_nonzero(edges):,} / {edges.size:,}")
print(f"Selected edge density: {100 * np.count_nonzero(edges) / edges.size:.2f}%")
print("Selection basis: visual balance between contour retention and texture suppression.")
print("Validation limit: no ground-truth edge labels; optimality is not established.")

**Reference — documentation:** [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — lower/upper thresholds and connectivity. The numerical selection and visual assessment above are specific to `uni.jpeg`, not prescribed by the documentation.

### 6. Investigate the effect of changing the lower and upper thresholds.

#### 6.1 Edge connectivity

**Connectivity** describes whether edge pixels form continuous paths. A **connected component** is a group linked by 8-neighbor paths. We report component count and largest component size (in pixels) as descriptive summaries, not accuracy measures: unrelated edges may merge, and legitimate objects may form separate components.

**Controlled comparison:** use the same `smoothed_gray`, Sobel aperture, and L2 setting. In the first row, vary only the lower threshold while fixing the upper at 150; in the second row, vary only the upper while fixing the lower at 50. The baseline (50, 150) appears in both rows. A **threshold sweep** means repeating the detector over specified threshold values.

In [ ]:
# Sweep one threshold at a time and summarize 8-connected edge components.
threshold_sweeps = [
    [(25, 150), (50, 150), (100, 150)],
    [(50, 75), (50, 150), (50, 225)]
]
connectivity_results = {}
fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True, sharey=True)
print(f"{'Lower':>6} {'Upper':>6} {'Edge pixels':>12} {'Density %':>10} {'Components':>12} {'Largest (px)':>13}")
for row, pairs in enumerate(threshold_sweeps):
    for column, (low, high) in enumerate(pairs):
        trial_edges = cv2.Canny(smoothed_gray, low, high,
                               apertureSize=sobel_aperture, L2gradient=use_l2_gradient)
        count, labels, stats, centroids = cv2.connectedComponentsWithStats(
            trial_edges, connectivity=8
        )
        component_sizes = stats[1:, cv2.CC_STAT_AREA]
        largest_component = int(component_sizes.max()) if component_sizes.size else 0
        trial_count = int(np.count_nonzero(trial_edges))
        connectivity_results[(low, high)] = trial_edges
        print(f"{low:6d} {high:6d} {trial_count:12,d} {100 * trial_count / trial_edges.size:10.2f} {count - 1:12,d} {largest_component:13,d}")
        ax = axes[row, column]
        ax.imshow(trial_edges, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
        ax.set_title(f"Lower={low}, upper={high}")
        ax.set_xlabel("Column (pixels)")
        ax.set_ylabel("Row (pixels)")
fig.suptitle("Threshold sweeps: top — upper fixed at 150; bottom — lower fixed at 50")
plt.tight_layout()
plt.show()

**Reading the output:** `connectedComponentsWithStats` returns labels and component statistics. Label 0 is background, so `count - 1` excludes it; `stats[1:, cv2.CC_STAT_AREA]` selects foreground component sizes. Size counts pixels, not geometric contour length. `centroids` are component mean positions and are unused here.

**Expected mechanism:** Lowering the lower threshold admits weaker links to existing strong seeds and can join fragments; raising it can remove links and fragment or delete contours. Lowering the upper threshold introduces more strong seeds and can retain previously unseeded candidate components; raising it can remove components lacking a sufficiently strong seed. These changes can increase or decrease the total component count, so that count alone cannot prove improved continuity.

**References — documentation:** [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — hysteresis; [OpenCV: Structural Analysis and Shape Descriptors](https://docs.opencv.org/4.x/d3/dc0/group__imgproc__shape.html) — `connectedComponentsWithStats` and `CC_STAT_AREA`.

**Observed results:** With upper=150, raising the lower threshold from 25 to 100 reduces edge pixels from 53,098 to 33,531, raises component count from 594 to 969, and reduces the largest component from 2,319 to 834 pixels, consistent with loss of weak connecting segments. With lower=50, raising the upper threshold from 75 to 225 reduces component count from 1,832 to 321 while the largest remains 1,547 pixels: many components lose their seeds, rather than the remaining contours necessarily becoming better connected.

#### 6.2 Edge localization

**Localization** is how closely a detected edge position matches the true intensity boundary. Here, smoothing and derivative settings remain fixed: changing hysteresis thresholds selects among the same thinned candidate positions rather than recomputing their locations. It can remove a weak contour or retain a nearby competing response, affecting which boundary is visible.

A **pixel mask** is a Boolean array marking selected coordinates. Compare a relaxed threshold pair with a stricter pair: count shared coordinates and strict-only coordinates. A zero strict-only count supports subset selection for these runs; it does not measure localization error against a true boundary.

For a magnified view, a **region of interest (ROI)** selects rows 160–299 and columns 250–429 using end-exclusive slices. Overlay colors distinguish shared edge pixels (white), pixels retained only at lower=25 (cyan), and pixels retained only at lower=100 (magenta).

In [ ]:
# Compare edge coordinates under stricter thresholds and magnify a fixed region.
ordered_threshold_pairs = [
    ((25, 150), (50, 150)), ((50, 150), (100, 150)),
    ((50, 75), (50, 150)), ((50, 150), (50, 225))
]
print(f"{'Relaxed -> stricter':>30} {'Shared pixels':>14} {'Stricter-only':>14}")
for relaxed_pair, strict_pair in ordered_threshold_pairs:
    relaxed_mask = connectivity_results[relaxed_pair] > 0
    strict_mask = connectivity_results[strict_pair] > 0
    shared_count = np.count_nonzero(relaxed_mask & strict_mask)
    strict_only_count = np.count_nonzero(strict_mask & ~relaxed_mask)
    print(f"{str(relaxed_pair) + ' -> ' + str(strict_pair):>30} {shared_count:14,d} {strict_only_count:14,d}")
roi_rows, roi_columns = slice(160, 300), slice(250, 430)
relaxed_roi = connectivity_results[(25, 150)][roi_rows, roi_columns] > 0
strict_roi = connectivity_results[(100, 150)][roi_rows, roi_columns] > 0
coordinate_overlay = np.zeros((*relaxed_roi.shape, 3), dtype=np.uint8)
coordinate_overlay[relaxed_roi & strict_roi] = [255, 255, 255]
coordinate_overlay[relaxed_roi & ~strict_roi] = [0, 255, 255]
coordinate_overlay[strict_roi & ~relaxed_roi] = [255, 0, 255]
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(gray[roi_rows, roi_columns], cmap="gray", vmin=0, vmax=255, interpolation="nearest")
axes[0].set_title("Grayscale ROI")
axes[1].imshow(coordinate_overlay, interpolation="nearest")
axes[1].set_title("White: shared; cyan: relaxed-only; magenta: strict-only")
for ax in axes:
    ax.set_xlabel("Column within ROI (pixels)")
    ax.set_ylabel("Row within ROI (pixels)")
plt.tight_layout()
plt.show()

**Interpretation:** In these controlled runs, stricter settings retain a subset of the relaxed edge coordinates (see the stricter-only counts). Shared responses remain at identical pixel positions, while weaker segments disappear. This is selection and loss of coverage, not evidence that the retained edges move toward a true boundary. Smoothing scale, derivative estimation, and non-maximum suppression govern the candidate positions. Without annotated boundaries, we cannot report localization error or claim that higher thresholds improve positional accuracy.

**Reference — further reading:** [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — directional suppression followed by hysteresis selection.

#### 6.3 False-positive detections

A **false positive (FP)** is a detected edge that does not match a target edge in a labeled reference. A **false negative (FN)** is a target edge that the detector misses. **Ground truth** is the annotated reference defining which boundaries count as targets.

Texture (repeated fine intensity changes) and shadows can produce genuine image gradients without indicating the object boundaries of interest. They are not automatically false positives: the judgment depends on the target definition. Here, visual inspection focuses on recognizable building and path boundaries; no pixel-level ground truth is available.

Lower thresholds can admit noise-driven responses and additional texture; higher thresholds suppress many weak responses but can also discard valid low-contrast boundaries. The following fixed ROI samples vegetation and foreground texture. It varies only the upper threshold (75, 150, 225), keeping the lower at 50 and all preprocessing unchanged. Counts of added or removed pixels describe sensitivity, not verified FP/FN counts.

In [ ]:
# Inspect textured foreground responses without treating extra edge pixels as confirmed errors.
texture_rows, texture_columns = slice(330, 460), slice(50, 300)
fp_pairs = [(50, 75), (50, 150), (50, 225)]
baseline_mask = connectivity_results[(50, 150)] > 0
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=True, sharey=True)
axes[0].imshow(gray[texture_rows, texture_columns], cmap="gray", vmin=0, vmax=255, interpolation="nearest")
axes[0].set_title("Grayscale texture ROI")
print("Texture ROI: rows 330–459, columns 50–299")
print(f"{'Lower':>6} {'Upper':>6} {'ROI edges':>10} {'ROI density %':>14} {'Added vs baseline':>18} {'Removed vs baseline':>20}")
for ax, pair in zip(axes[1:], fp_pairs):
    trial_mask = connectivity_results[pair] > 0
    roi_mask = trial_mask[texture_rows, texture_columns]
    added_pixels = np.count_nonzero(trial_mask & ~baseline_mask)
    removed_pixels = np.count_nonzero(baseline_mask & ~trial_mask)
    print(f"{pair[0]:6d} {pair[1]:6d} {np.count_nonzero(roi_mask):10,d} {100 * np.mean(roi_mask):14.2f} {added_pixels:18,d} {removed_pixels:20,d}")
    ax.imshow(roi_mask, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(f"Edges: lower={pair[0]}, upper={pair[1]}")
for ax in axes:
    ax.set_xlabel("Column within ROI (pixels)")
    ax.set_ylabel("Row within ROI (pixels)")
plt.tight_layout()
plt.show()
print("Added/removed counts use the full image; ROI counts use the displayed crop.")
print("No annotated reference: FP counts, FN counts, and false-positive rates are not measured.")

**Interpretation:** Compare the cropped edge maps against the grayscale texture: stricter settings remove many responses in vegetation, but a cleaner-looking result alone does not establish fewer errors. The full-image threshold sweep also shows loss of weaker contours. Thus threshold selection balances additional responses against missing useful structure; the selected (50, 150) pair is a visual compromise for this image.

**Task 2 finding:** Thresholds change edge survival and connectivity. With fixed smoothing and derivatives, shared edge coordinates remain unchanged. Without labeled target boundaries, localization error and false-positive rates cannot be quantified from edge density or component counts.

**Reference — further reading:** [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — noise sensitivity and threshold-dependent retention. The ROI assessment is a qualitative analysis of this image.

## Task 3: Gaussian Filtering and Comparison

### 1. Apply a Gaussian filter with standard deviation σ = 2√2.

A **Gaussian filter** smooths the image using a normalized, bell-shaped weighting kernel. **Standard deviation** $\sigma$ sets its spatial spread in pixels; **variance** is $\sigma^2$. Here $\sigma=2\sqrt{2}\approx2.8284$ pixels and variance is 8 pixels².

Use the original Task 1 grayscale image, not the Canny-smoothed input. For a reproducible finite kernel, choose radius $r=\lceil3\sigma\rceil$ and odd side length $k=2r+1$, giving 19 × 19. The **ceiling** operation rounds upward; this 3σ support rule is an implementation choice, not an assignment requirement. **Truncation** means omitting the Gaussian's infinite tails.

`astype(np.float64)` converts the stored intensities to 64-bit floating point without rescaling 0–255. It avoids rounding each filtering result back to `uint8`, which would obscure the later comparison. Set equal `sigmaX` and `sigmaY` for equal smoothing in both axes, and explicitly use reflected borders.

In [ ]:
# Apply one Gaussian pass to the original grayscale image using explicit support and precision.
gray_float = gray.astype(np.float64)
sigma_single = 2.0 * np.sqrt(2.0)
support_sigmas = 3.0
radius_single = int(np.ceil(support_sigmas * sigma_single))
kernel_single = 2 * radius_single + 1
gaussian_border = cv2.BORDER_REFLECT_101
gaussian_single = cv2.GaussianBlur(
    gray_float, (kernel_single, kernel_single),
    sigmaX=sigma_single, sigmaY=sigma_single, borderType=gaussian_border
)
print("Input source: original Task 1 grayscale image")
print(f"Standard deviation: 2*sqrt(2) = {sigma_single:.8f} pixels")
print(f"Variance: {sigma_single**2:.4f} pixels squared")
print(f"Support rule: radius = ceil({support_sigmas:g} * sigma)")
print(f"Kernel radius: {radius_single} pixels; kernel size: {kernel_single} x {kernel_single}")
print("Border handling: BORDER_REFLECT_101")
print(f"Input/output data type: {gray_float.dtype} / {gaussian_single.dtype}")
print(f"Output shape: {gaussian_single.shape}")
print(f"Observed output range: [{gaussian_single.min():.4f}, {gaussian_single.max():.4f}]")

**Interpretation:** This produces one smoothed floating-point image with unchanged spatial dimensions. Fractional values are retained for the subsequent two-pass comparison; the original `gray` array remains unchanged.

**Reference — documentation:** [OpenCV: Image Filtering](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html), `GaussianBlur` and `getGaussianKernel` — kernel size, standard deviations, normalized coefficients, and border handling.

### 2. Apply a Gaussian filter twice, using σ = 2 for each operation.

A **pass** is one application of a filter. Start from `gray_float`, the original Task 1 grayscale image converted to floating point, then filter the first result again. Filtering the original independently twice would not accumulate smoothing.

For ideal continuous Gaussian convolution, **variances add**:

$$\sigma_{\mathrm{effective}}^2=\sigma_1^2+\sigma_2^2=2^2+2^2=8,\qquad \sigma_{\mathrm{effective}}=2\sqrt{2}.$$

The standard deviations themselves do not add. This predicts agreement with Step 1 in the ideal continuous setting. Finite sampled kernels, borders, and numerical arithmetic can produce practical differences, investigated in Step 6.

Apply the same support rule $r=\lceil3\sigma\rceil$, giving a 13 × 13 kernel for each pass. Keep `float64` throughout and the same border rule as Step 1. The composed finite kernel has a wider support than either individual kernel; it is not simply a single 13 × 13 filter.

In [ ]:
# Apply two successive sigma-2 Gaussian passes to the original floating-point grayscale image.
sigma_pass = 2.0
radius_pass = int(np.ceil(support_sigmas * sigma_pass))
kernel_pass = 2 * radius_pass + 1
gaussian_pass1 = cv2.GaussianBlur(
    gray_float, (kernel_pass, kernel_pass),
    sigmaX=sigma_pass, sigmaY=sigma_pass, borderType=gaussian_border
)
gaussian_twice = cv2.GaussianBlur(
    gaussian_pass1, (kernel_pass, kernel_pass),
    sigmaX=sigma_pass, sigmaY=sigma_pass, borderType=gaussian_border
)
sigma_effective = np.sqrt(sigma_pass**2 + sigma_pass**2)
print("Pass 1 input: original Task 1 grayscale image converted to float64")
print("Pass 2 input: output of pass 1")
print(f"Sigma per pass: {sigma_pass:.1f} pixels; variance per pass: {sigma_pass**2:.1f} pixels squared")
print(f"Kernel per pass: {kernel_pass} x {kernel_pass}; radius: {radius_pass} pixels")
print(f"Ideal combined variance: {2 * sigma_pass**2:.1f} pixels squared")
print(f"Ideal effective sigma: {sigma_effective:.8f} pixels")
print("Border handling for both passes: BORDER_REFLECT_101")
for label, result in (("Pass 1", gaussian_pass1), ("Pass 2", gaussian_twice)):
    print(f"{label}: shape={result.shape}, dtype={result.dtype}, range=[{result.min():.4f}, {result.max():.4f}]")

**Interpretation:** `gaussian_twice` stores the sequential two-pass result. Its ideal effective standard deviation matches Step 1; numerical equality of the actual images remains to be tested in the following steps.

**Reference — documentation:** [OpenCV: Image Filtering, GaussianBlur](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html) — Gaussian filtering parameters. The variance-addition relation applies to ideal Gaussian convolution, not an exact guarantee for these finite implementations.

### 3. Display both resulting images.

Use the same grayscale colormap and **display limits** (`vmin=0`, `vmax=255`) for both floating-point results. The conversion to `float64` did not rescale the intensities to 0–1. Fixed limits avoid independent contrast stretching that could distort the visual comparison. Displaying the arrays does not round or overwrite their values.

In [ ]:
# Display the single-pass and two-pass Gaussian results using identical intensity scales.
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
gaussian_display_results = [
    (gaussian_single, f"One pass: sigma = 2*sqrt(2) ≈ {sigma_single:.4f}"),
    (gaussian_twice, "Two successive passes: sigma = 2 each")
]
for ax, (result, title) in zip(axes, gaussian_display_results):
    ax.imshow(result, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
    ax.set_title(title)
    ax.set_xlabel("Column (pixels)")
    ax.set_ylabel("Row (pixels)")
fig.suptitle("Gaussian filtering comparison — common intensity range: 0–255")
plt.tight_layout()
plt.show()

**Observation:** At this display scale, the two results appear very similar: both soften fine texture and edges while retaining the overall scene structure. Visual similarity does not establish pixel-wise equality; the next step computes their absolute difference.

**Reference — documentation:** [Matplotlib: imshow](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html) — scalar image display, colormaps, and intensity limits.

### 4. Compute the absolute pixel-wise difference between the two filtered images.

A **pixel-wise difference** compares values at matching row and column coordinates. The **absolute value** removes the sign, so the difference measures the size of a disagreement rather than which result is brighter:

$$D(x,y)=\left|I_{\mathrm{single}}(x,y)-I_{\mathrm{twice}}(x,y)\right|.$$

Retain `float64` values during subtraction: converting to `uint8` would discard fractional differences, and unsigned subtraction can wrap around. **Mean absolute difference** averages $D$ across all pixels; **maximum absolute difference** reports the largest disagreement. Both use the original 0–255 intensity units, not percentages.

A **tolerance** is a chosen numerical cutoff for reporting small differences. We additionally count values above $10^{-6}$ intensity units; this is a reporting convention, not a perceptual threshold.

In [ ]:
# Compute absolute differences in floating point and report their size and frequency.
absolute_difference = np.abs(gaussian_single - gaussian_twice)
difference_tolerance = 1e-6
exact_nonzero_count = np.count_nonzero(absolute_difference)
above_tolerance_count = np.count_nonzero(absolute_difference > difference_tolerance)
print(f"Difference shape: {absolute_difference.shape}; dtype: {absolute_difference.dtype}")
print("Difference units: intensity units on the original 0–255 scale")
print(f"Minimum absolute difference: {absolute_difference.min():.10f}")
print(f"Maximum absolute difference: {absolute_difference.max():.10f}")
print(f"Mean absolute difference: {absolute_difference.mean():.10f}")
print(f"Exactly nonzero differences: {exact_nonzero_count:,} / {absolute_difference.size:,}")
print(f"Reporting tolerance: {difference_tolerance:g} intensity units")
print(f"Differences above tolerance: {above_tolerance_count:,} ({100 * above_tolerance_count / absolute_difference.size:.2f}%)")

**Interpretation:** A large count of nonzero differences does not by itself mean a large discrepancy; floating-point arrays can differ by tiny amounts at many pixels. Use the reported magnitudes together with the difference image in the next step.

**Reference — documentation:** [NumPy: absolute](https://numpy.org/doc/stable/reference/generated/numpy.absolute.html) — element-wise absolute values.

### 5. Display the difference image and briefly explain the observed results.

**Display scaling** maps numerical values to colors without changing the stored difference array. The left panel uses the original 0–255 intensity scale; the right panel expands the display range to 0–the observed maximum difference. A **colorbar** identifies the numerical value represented by each display color. The expanded view reveals spatial patterns but visually exaggerates their small amplitude.

In [ ]:
# Show absolute differences on the original scale and an explicitly expanded scale.
max_difference = float(absolute_difference.max())
expanded_limit = max_difference if max_difference > 0 else 1.0
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
original_scale_plot = axes[0].imshow(
    absolute_difference, cmap="gray", vmin=0, vmax=255, interpolation="nearest"
)
axes[0].set_title("Absolute difference — original scale [0, 255]")
fig.colorbar(original_scale_plot, ax=axes[0], label="Absolute difference (intensity units)")
expanded_scale_plot = axes[1].imshow(
    absolute_difference, cmap="magma", vmin=0, vmax=expanded_limit, interpolation="nearest"
)
axes[1].set_title(f"Absolute difference — expanded scale [0, {expanded_limit:.4f}]")
fig.colorbar(expanded_scale_plot, ax=axes[1], label="Absolute difference (intensity units)")
for ax in axes:
    ax.set_xlabel("Column (pixels)")
    ax.set_ylabel("Row (pixels)")
plt.tight_layout()
plt.show()
print(f"Maximum difference as a percentage of the 255-unit range: {100 * max_difference / 255:.4f}%")
print("Expanded display changes only color mapping; difference values remain unchanged.")

**Observation:** The original-scale difference image appears nearly black because the maximum discrepancy is about 0.2166 intensity units (0.0849% of the full 255-unit range); the mean is about 0.02575. The expanded view reveals nonuniform differences around scene detail and intensity transitions. Bright regions in that view indicate relatively larger discrepancies, not differences approaching 255.

The ideal variance-addition relation explains the similar smoothing, but the implemented finite filters are not numerically identical. The following step investigates truncation, sampling, normalization, borders, and numerical precision separately.

**Reference — documentation:** [Matplotlib: imshow](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html) — colormaps and normalization limits.

### 6. Investigate why the implementations may produce non-zero differences.

#### 6.1 Finite kernel truncation

An ideal Gaussian has **infinite support**: its weights are nonzero arbitrarily far from the center. **Finite support** limits the offsets included in a practical kernel. **Truncation** omits the outer tails; the retained sampled weights are then normalized to sum to one.

Vary only the support multiplier $t$ in $r=\lceil t\sigma\rceil$, using 2, 3, and 4. Keep the input, `float64` precision, sigma values, normalized OpenCV kernels, and reflected border handling fixed. The single-pass and composed two-pass kernels lose different tails, so finite implementations need not obey the ideal variance-addition identity exactly.

The **interior** excludes a border margin equal to the larger effective radius: $\max(r_{single},2r_{pass})$. This removes dependence on extrapolated pixels for both filtering routes. Interior differences help distinguish finite-kernel effects from boundary effects; residual sampling and arithmetic effects remain.

In [ ]:
# Vary Gaussian support while keeping sigma, precision, normalization, and borders fixed.
truncation_results = []
print(f"{'Support':>8} {'Single k':>10} {'Pass k':>8} {'Margin':>8} {'Full mean':>13} {'Full max':>13} {'Interior mean':>15} {'Interior max':>14}")
for support in (2.0, 3.0, 4.0):
    single_radius = int(np.ceil(support * sigma_single))
    pass_radius = int(np.ceil(support * sigma_pass))
    single_size = 2 * single_radius + 1
    pass_size = 2 * pass_radius + 1
    single_trial = cv2.GaussianBlur(gray_float, (single_size, single_size),
                                  sigmaX=sigma_single, sigmaY=sigma_single,
                                  borderType=gaussian_border)
    first_trial = cv2.GaussianBlur(gray_float, (pass_size, pass_size),
                                 sigmaX=sigma_pass, sigmaY=sigma_pass,
                                 borderType=gaussian_border)
    twice_trial = cv2.GaussianBlur(first_trial, (pass_size, pass_size),
                                 sigmaX=sigma_pass, sigmaY=sigma_pass,
                                 borderType=gaussian_border)
    trial_difference = np.abs(single_trial - twice_trial)
    margin = max(single_radius, 2 * pass_radius)
    interior_difference = trial_difference[margin:-margin, margin:-margin]
    truncation_results.append((support, float(trial_difference.mean()),
                               float(interior_difference.mean())))
    print(f"{support:7.0f}σ {single_size:10d} {pass_size:8d} {margin:8d} {trial_difference.mean():13.8f} {trial_difference.max():13.8f} {interior_difference.mean():15.8f} {interior_difference.max():14.8f}")
print("Difference statistics are in original intensity units; interior margins are pixels.")

**Interpretation:** Enlarging support retains more of each Gaussian tail and tests how strongly the discrepancy depends on truncation. The baseline uses 3σ support. Differences remaining with wider support need not be rounding errors: sampling the continuous Gaussian on an integer grid also changes its composition behavior. This experiment varies support; it does not independently isolate every source of error.

**Reference — documentation:** [OpenCV: Image Filtering](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html), `getGaussianKernel` and `GaussianBlur` — finite Gaussian coefficients, normalization, kernel size, and standard deviation.

**Observed result:** Full-image mean absolute differences decrease from 0.28791 (2σ) to 0.02575 (3σ) and 0.000706 (4σ). The same trend occurs in the interior, supporting a substantial contribution from finite support in this setup rather than a purely boundary-driven discrepancy. Each support uses its own sufficient interior margin.

#### 6.2 Discrete sampling

**Sampling** evaluates a continuous function at selected positions. A **pixel grid** uses integer offsets with unit spacing. A sampled Gaussian uses weights proportional to $\exp[-n^2/(2\sigma^2)]$ at integer $n$, then normalizes their sum.

Continuous convolution integrates over all positions, whereas **discrete convolution** sums over grid points. Sampling after continuous convolution is generally not identical to convolving already sampled kernels. Thus two sampled Gaussian kernels need not compose to exactly the sampled Gaussian with summed nominal variances.

Test the one-dimensional kernels directly to avoid image-border effects; the two-dimensional Gaussian is separable, meaning its kernel is a product of one-dimensional kernels. Use wide 8σ support to make omitted tails very small, and `float64` arithmetic. Include σ=0.5 and σ=1 as illustrative cases, plus the assignment's σ=2. `np.convolve(..., mode="full")` retains the complete composed finite kernel. Zero-padding aligns both kernel centers on the same integer-offset grid.

In [ ]:
# Compare composed and directly sampled Gaussian kernels with negligible tail truncation.
sampling_kernel_results = []
print(f"{'Sigma per pass':>15} {'Composed sum':>15} {'Direct sum':>15} {'Max coefficient gap':>21}")
for sample_sigma in (0.5, 1.0, 2.0):
    sample_radius = int(np.ceil(8 * sample_sigma))
    sample_offsets = np.arange(-sample_radius, sample_radius + 1, dtype=np.float64)
    sampled_kernel = np.exp(-sample_offsets**2 / (2 * sample_sigma**2))
    sampled_kernel /= sampled_kernel.sum()
    composed_kernel = np.convolve(sampled_kernel, sampled_kernel, mode="full")
    direct_sigma = np.sqrt(2) * sample_sigma
    direct_radius = int(np.ceil(8 * direct_sigma))
    direct_offsets = np.arange(-direct_radius, direct_radius + 1, dtype=np.float64)
    direct_kernel = np.exp(-direct_offsets**2 / (2 * direct_sigma**2))
    direct_kernel /= direct_kernel.sum()
    common_radius = max(2 * sample_radius, direct_radius)
    composed_aligned = np.pad(composed_kernel, common_radius - 2 * sample_radius)
    direct_aligned = np.pad(direct_kernel, common_radius - direct_radius)
    coefficient_gap = float(np.max(np.abs(composed_aligned - direct_aligned)))
    sampling_kernel_results.append((sample_sigma, coefficient_gap))
    print(f"{sample_sigma:15.1f} {composed_kernel.sum():15.12f} {direct_kernel.sum():15.12f} {coefficient_gap:21.12e}")
print("Kernel coefficients are dimensionless weights, not image intensity differences.")

**Interpretation:** A clear coefficient discrepancy for narrow σ demonstrates that normalized sampled Gaussian kernels do not generally obey the continuous composition identity. For σ=2, the wide-support discrepancy may be near floating-point precision; this does not justify attributing the baseline image difference mainly to sampling. The earlier support sweep provides stronger evidence for truncation in the assignment's 3σ setup. This experiment greatly reduces truncation but does not eliminate finite-support and arithmetic effects entirely.

**References — documentation:** [NumPy: convolve](https://numpy.org/doc/stable/reference/generated/numpy.convolve.html) — discrete convolution; [OpenCV: Image Filtering](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html), `getGaussianKernel` — sampled, normalized Gaussian coefficients.

#### 6.3 Kernel normalization

**Normalization** divides every kernel coefficient by the sum of its coefficients:

$$K_{normalized}(i,j)=\frac{K_{raw}(i,j)}{\sum_{u,v}K_{raw}(u,v)}.$$

The normalized weights sum to 1. This sum is the filter's **DC gain**, meaning its response multiplier for a constant image. A unit DC gain preserves constant intensity when boundary extension also preserves the constant. It does not mean every image's global mean is exactly preserved under every boundary rule.

OpenCV's Gaussian coefficients are normalized. Because a Gaussian is **separable**, its two-dimensional kernel is the outer product of two one-dimensional kernels; the 2D sum is the product of their sums. The **outer product** multiplies every coefficient of one vector by every coefficient of the other.

After truncation, normalizing different retained supports changes the relative scaling of their coefficients. Both resulting filters can sum to 1 yet have different shapes. The following check tests the assignment's normalized kernels and a constant-image response, not a claim that missing normalization caused the observed discrepancy.

In [ ]:
# Verify Gaussian kernel sums and constant-intensity preservation for both filtering routes.
normal_kernel_single = cv2.getGaussianKernel(kernel_single, sigma_single, cv2.CV_64F).ravel()
normal_kernel_pass = cv2.getGaussianKernel(kernel_pass, sigma_pass, cv2.CV_64F).ravel()
normal_kernel_composed = np.convolve(normal_kernel_pass, normal_kernel_pass, mode="full")
print(f"Single-pass 1D kernel sum: {normal_kernel_single.sum():.16f}")
print(f"Per-pass 1D kernel sum: {normal_kernel_pass.sum():.16f}")
print(f"Composed two-pass 1D kernel sum: {normal_kernel_composed.sum():.16f}")
print(f"Single-pass 2D kernel sum: {np.outer(normal_kernel_single, normal_kernel_single).sum():.16f}")
print(f"Per-pass 2D kernel sum: {np.outer(normal_kernel_pass, normal_kernel_pass).sum():.16f}")
constant_image = np.full((64, 64), 100.0, dtype=np.float64)
constant_single = cv2.GaussianBlur(constant_image, (kernel_single, kernel_single),
                                  sigmaX=sigma_single, sigmaY=sigma_single,
                                  borderType=gaussian_border)
constant_first = cv2.GaussianBlur(constant_image, (kernel_pass, kernel_pass),
                                 sigmaX=sigma_pass, sigmaY=sigma_pass,
                                 borderType=gaussian_border)
constant_twice = cv2.GaussianBlur(constant_first, (kernel_pass, kernel_pass),
                                 sigmaX=sigma_pass, sigmaY=sigma_pass,
                                 borderType=gaussian_border)
print("Constant input: 100 intensity units; border: BORDER_REFLECT_101")
print(f"Single-pass maximum deviation from 100: {np.max(np.abs(constant_single - 100)):.12e}")
print(f"Two-pass maximum deviation from 100: {np.max(np.abs(constant_twice - 100)):.12e}")

**Interpretation:** Kernel sums near 1 and constant-image deviations near floating-point precision confirm normalization and constant preservation for these settings. They do not prove that the single and composed kernels match coefficient by coefficient. Renormalization of differently truncated kernels contributes to their shape differences, while failure to normalize would additionally change constant intensity.

**Reference — documentation:** [OpenCV: Image Filtering](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html), `getGaussianKernel` — normalized Gaussian coefficients and use in separable filtering.

#### 6.4 Boundary handling

**Boundary handling** specifies values outside the image when a filter neighborhood extends beyond its borders. **Extrapolation** estimates these unavailable values from a rule; they are not observed scene pixels.

| OpenCV rule | Extension outside the image |
| --- | --- |
| `BORDER_REFLECT_101` | Reflect values without repeating the boundary pixel. |
| `BORDER_REPLICATE` | Repeat the nearest boundary pixel. |
| `BORDER_CONSTANT` | Use zero for GaussianBlur's constant border. |

Keep the baseline kernels, sigma values, and `float64` input unchanged. Apply the selected rule on every pass. In the two-pass route, the second pass extrapolates the already filtered image; this can differ from filtering one consistently extended original image. The effect depends on the rule; reflection can be compatible with composition, so repeated filtering does not necessarily introduce an extra boundary discrepancy.

Define the **border band** as all pixels within 12 pixels of an image edge, using $\max(9,2\times6)=12$, the larger effective kernel radius. The **interior** is the complement and cannot depend on extrapolated values for these kernels.

In [ ]:
# Compare Gaussian routes under three boundary rules, separating border and interior errors.
boundary_margin = max(radius_single, 2 * radius_pass)
border_band = np.ones(gray.shape, dtype=bool)
border_band[boundary_margin:-boundary_margin, boundary_margin:-boundary_margin] = False
boundary_modes = [
    ("REFLECT_101", cv2.BORDER_REFLECT_101),
    ("REPLICATE", cv2.BORDER_REPLICATE),
    ("CONSTANT_ZERO", cv2.BORDER_CONSTANT)
]
boundary_differences = {}
print(f"Border band width: {boundary_margin} pixels")
print(f"Border pixels: {np.count_nonzero(border_band):,}; interior pixels: {np.count_nonzero(~border_band):,}")
print(f"{'Border rule':>15} {'Border mean':>14} {'Border max':>14} {'Interior mean':>15} {'Interior max':>14}")
for boundary_name, boundary_mode in boundary_modes:
    boundary_single = cv2.GaussianBlur(gray_float, (kernel_single, kernel_single),
                                      sigmaX=sigma_single, sigmaY=sigma_single,
                                      borderType=boundary_mode)
    boundary_first = cv2.GaussianBlur(gray_float, (kernel_pass, kernel_pass),
                                     sigmaX=sigma_pass, sigmaY=sigma_pass,
                                     borderType=boundary_mode)
    boundary_twice = cv2.GaussianBlur(boundary_first, (kernel_pass, kernel_pass),
                                     sigmaX=sigma_pass, sigmaY=sigma_pass,
                                     borderType=boundary_mode)
    boundary_difference = np.abs(boundary_single - boundary_twice)
    boundary_differences[boundary_name] = boundary_difference
    border_values = boundary_difference[border_band]
    interior_values = boundary_difference[~border_band]
    print(f"{boundary_name:>15} {border_values.mean():14.8f} {border_values.max():14.8f} {interior_values.mean():15.8f} {interior_values.max():14.8f}")
print("All statistics are absolute differences in original intensity units.")

**Interpretation:** Compare the border statistics across rules while checking that interior statistics remain the same. Zero extension creates an artificial intensity drop outside the image, and repeated re-extension of the intermediate image can increase boundary discrepancies. Repeating boundary values can also alter the two-pass behavior. These are consequences of the chosen extension model, not evidence that the continuous Gaussian identity is false. The reflected baseline still has finite-kernel differences throughout the image.

**Reference — documentation:** [OpenCV: Image Filtering](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html) — extrapolation and Gaussian filtering; [OpenCV: Basic Operations on Images](https://docs.opencv.org/4.x/d3/df2/tutorial_py_basic_ops.html) — border extension rules.

**Observed result:** Mean border discrepancies are approximately 0.01767 (reflection), 0.11627 (replication), and 2.65574 (zero extension). The interior mean is 0.02650613 for every rule. This localizes the effect of changing the extension rule to the expected border region in this experiment.

#### 6.5 Numerical precision

**Floating-point precision** limits how accurately fractional values and arithmetic results can be represented. `float32` stores 32 bits per value; `float64` stores 64 and generally reduces rounding error. More precision does not remove finite-kernel or sampling differences.

**Rounding** replaces a result with a nearby representable value. **Quantization** restricts values to discrete levels; `uint8` filtering stores integer intensity values after each pass, discarding fractional information. Two successive integer-output passes can therefore differ from a single pass even beyond kernel-shape effects. Optimized integer implementations may also use different coefficient arithmetic.

Keep kernel sizes, sigma values, and border handling fixed. Compare each route with its corresponding `float64` result as a numerical reference, not an exact ground truth. Convert results to `float64` before subtraction to avoid unsigned wraparound.

In [ ]:
# Compare output precision while keeping Gaussian kernels and border handling fixed.
precision_results = {}
print(f"{'Input dtype':>12} {'Route mean gap':>16} {'Route max gap':>16} {'Single vs f64 max':>19} {'Twice vs f64 max':>18}")
for precision_dtype in (np.float64, np.float32, np.uint8):
    precision_input = gray.astype(precision_dtype)
    precision_single = cv2.GaussianBlur(precision_input, (kernel_single, kernel_single),
                                       sigmaX=sigma_single, sigmaY=sigma_single,
                                       borderType=gaussian_border)
    precision_first = cv2.GaussianBlur(precision_input, (kernel_pass, kernel_pass),
                                      sigmaX=sigma_pass, sigmaY=sigma_pass,
                                      borderType=gaussian_border)
    precision_twice = cv2.GaussianBlur(precision_first, (kernel_pass, kernel_pass),
                                      sigmaX=sigma_pass, sigmaY=sigma_pass,
                                      borderType=gaussian_border)
    single_as_float = precision_single.astype(np.float64)
    twice_as_float = precision_twice.astype(np.float64)
    precision_gap = np.abs(single_as_float - twice_as_float)
    single_reference_gap = np.max(np.abs(single_as_float - gaussian_single))
    twice_reference_gap = np.max(np.abs(twice_as_float - gaussian_twice))
    precision_name = np.dtype(precision_dtype).name
    precision_results[precision_name] = (precision_single, precision_twice)
    print(f"{precision_name:>12} {precision_gap.mean():16.10f} {precision_gap.max():16.10f} {single_reference_gap:19.10f} {twice_reference_gap:18.10f}")
print("All gaps are in original intensity units. float64 is a comparison reference, not exact arithmetic.")

**Interpretation:** Compare the reference gaps to distinguish numerical representation effects from the baseline single-versus-two-pass discrepancy. Integer results include per-pass quantization and possible optimized integer arithmetic, so they do not isolate floating-point rounding alone. A smaller single-versus-two-pass gap after quantization would not establish greater accuracy: rounding can hide fractional discrepancies.

**Task 3 finding:** The outputs are visually similar but not identical. Wider kernel support sharply reduces their discrepancy. Discrete sampling can violate the continuous composition identity, especially for narrow Gaussians; for σ=2 with wide support, the measured kernel gap is near arithmetic precision. Both kernels are normalized, border rules affect the boundary region, and reduced numerical precision adds representation effects. These mechanisms interact rather than forming an independently additive error budget.

**References — documentation:** [NumPy: Data types](https://numpy.org/doc/stable/user/basics.types.html) — integer and floating-point representations; [OpenCV: Image Filtering](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html) — supported data depths and filtering parameters.

**Observed result:** Maximum deviations of the float32 routes from their float64 references are below 0.00008 intensity units. The uint8 routes differ by as much as about 1.12 units from their references, and their mutual maximum gap is 1 unit. Thus the floating-point rounding effect is small relative to the baseline 3σ discrepancy here, while integer storage has a larger effect.

## Task 4: Salt-and-Pepper Noise Removal

### 1. Read uni.jpeg and convert it to grayscale.

Reload the original image to begin this experiment with an unfiltered reference. A **reference image** is the baseline against which later noisy and filtered results will be compared; here it is the decoded original, not a perfectly noise-free measurement of the scene.

**Grayscale** stores one intensity value per pixel. **BGR** means blue, green, red: the channel order returned by `cv2.imread(..., cv2.IMREAD_COLOR)`. `cv2.cvtColor` converts between color representations; `cv2.COLOR_BGR2GRAY` specifies the matching conversion:

$$Y = 0.299R + 0.587G + 0.114B.$$

The unequal weights approximate perceived brightness contributions. The conversion preserves spatial dimensions but loses color information. The output uses **uint8** (unsigned 8-bit integers), representing 256 levels from 0 (black) to 255 (white). Its **observed range** describes the values actually present, rather than every value allowed by the type.

| Name / operation | Meaning |
| --- | --- |
| `task4_image_path` | Path to the original JPEG input. |
| `task4_bgr` | Newly decoded three-channel color array. |
| `clean_gray` | Unfiltered grayscale reference for the noise-removal experiments. “Clean” distinguishes it from the synthetic noise added later. |
| `str(...)` | Converts the Path object into a filename string for OpenCV. |
| `is None` | Detects a failed image read before attempting conversion. |
| `np.array_equal` | Checks that shape and every pixel value match the Task 1 reference. |

We explicitly read color and then convert, matching Task 1. Direct grayscale decoding can use codec-specific conversion and need not give identical values.

In [31]:
# Reload the original image; do not use a Gaussian-filtered result from Task 3.
task4_image_path = DATA_DIR / "uni.jpeg"
task4_bgr = cv2.imread(str(task4_image_path), cv2.IMREAD_COLOR)
if task4_bgr is None:
    raise FileNotFoundError(f"Unable to read image: {task4_image_path}")
clean_gray = cv2.cvtColor(task4_bgr, cv2.COLOR_BGR2GRAY)

assert clean_gray.ndim == 2 and clean_gray.dtype == np.uint8
assert clean_gray.shape == task4_bgr.shape[:2]
assert np.array_equal(clean_gray, gray), "Reloaded reference differs from Task 1."
print(f"Input file: {task4_image_path}")
print("Read mode: cv2.IMREAD_COLOR; channel order: BGR")
print(f"Loaded color shape (height, width, channels): {task4_bgr.shape}")
print("Conversion: cv2.COLOR_BGR2GRAY; Y = 0.299R + 0.587G + 0.114B")
print(f"Reference shape (height, width): {clean_gray.shape}")
print("Grayscale channels: 1")
print(f"Reference dtype: {clean_gray.dtype}; representable range: [0, 255]")
print(f"Observed intensity range: [{clean_gray.min()}, {clean_gray.max()}]")
print(f"Spatial pixels: {clean_gray.size:,}")
print(f"Reference pixel-data memory: {clean_gray.nbytes:,} bytes")
print(f"Spatial dimensions preserved: {clean_gray.shape == task4_bgr.shape[:2]}")
print(f"Pixel-for-pixel match with Task 1: {np.array_equal(clean_gray, gray)}")
print("Reference status: original grayscale; no synthetic noise or smoothing applied.")

Input file: /workspace/scratch/6dec89157306/ComputerVision/Assignments/Assign_1/data/uni.jpeg
Read mode: cv2.IMREAD_COLOR; channel order: BGR
Loaded color shape (height, width, channels): (476, 645, 3)
Conversion: cv2.COLOR_BGR2GRAY; Y = 0.299R + 0.587G + 0.114B
Reference shape (height, width): (476, 645)
Grayscale channels: 1
Reference dtype: uint8; representable range: [0, 255]
Observed intensity range: [0, 255]
Spatial pixels: 307,020
Reference pixel-data memory: 307,020 bytes
Spatial dimensions preserved: True
Pixel-for-pixel match with Task 1: True
Reference status: original grayscale; no synthetic noise or smoothing applied.


**Observation:** The reloaded reference has shape (476, 645), one channel, dtype `uint8`, and observed range [0, 255]. Its pixel data occupies 307,020 bytes. It matches Task 1 pixel for pixel. Later noise generation must modify a copy so that this reference remains available for comparison.

**References — official documentation:**
- [OpenCV: Image file reading and writing](https://docs.opencv.org/4.x/d4/da8/group__imgcodecs.html), `imread` — loading, BGR order, failure behavior, and codec-specific grayscale conversion.
- [OpenCV: Color conversions, RGB ↔ GRAY](https://docs.opencv.org/4.x/de/d25/imgproc_color_conversions.html) — weighted grayscale formula and conversion flags.

### 2. Add 30% salt-and-pepper noise.

**Salt-and-pepper noise** is an **impulse noise** model: selected pixels are replaced by extreme intensities, producing white (salt, 255) and black (pepper, 0) specks. Replacement overwrites the original value rather than adding a small intensity offset.

**Noise density** $d$ is the fraction of spatial pixels selected for replacement. With $N=H\times W$, select $n=\mathrm{round}(dN)$ distinct locations. At $d=0.30$, this image has exactly 92,106 selected pixels. Use equal numbers of salt and pepper assignments (46,053 each), randomly distributed over these locations. For an odd selected count, the two groups differ by one pixel.

**Sampling without replacement** means each location is selected at most once. A **seed** initializes a pseudorandom number generator, allowing the same realization to be reproduced with the same code and NumPy environment. A **realization** is one specific random arrangement of noise; the seed does not make this arrangement optimal or representative of every possible arrangement.

| Name / operation | Meaning |
| --- | --- |
| `add_salt_pepper` | Reusable function returning a noisy copy and a selection mask. |
| `density` | Fraction of pixels selected, between 0 and 1. |
| `np.random.default_rng(seed)` | Creates a local random generator without changing NumPy's global random state. |
| `rng.choice(..., replace=False)` | Uniformly selects distinct flattened pixel indices. |
| `.flat` | Accesses the array through a one-dimensional sequence of pixel locations. |
| `.copy()` | Allocates a separate image so the reference remains unchanged. |
| `selected_mask` | Boolean array: True at selected locations, False elsewhere. |
| `np.count_nonzero` | Counts True values when applied to a Boolean condition. |

The **selected fraction** and **changed-value fraction** differ when a pixel already equals its assigned extreme. Such a pixel was selected and overwritten but its stored value did not change. Counting every 0 or 255 in the noisy image also counts extremes already present in the original, so it does not measure noise density.

In [32]:
# Select unique pixel locations and preserve the original reference.
def add_salt_pepper(image, density, seed=42):
    """Replace round(density * N) distinct uint8 grayscale pixels with 0 or 255."""
    if image.ndim != 2 or image.dtype != np.uint8:
        raise ValueError("Expected a two-dimensional uint8 grayscale image.")
    if not np.isfinite(density) or not 0 <= density <= 1:
        raise ValueError("density must be finite and between 0 and 1.")
    rng = np.random.default_rng(seed)
    n_selected = int(round(density * image.size))
    selected_indices = rng.choice(image.size, size=n_selected, replace=False)
    n_pepper = n_selected // 2
    noisy = image.copy()
    noisy.flat[selected_indices[:n_pepper]] = 0
    noisy.flat[selected_indices[n_pepper:]] = 255
    selected_mask = np.zeros(image.shape, dtype=bool)
    selected_mask.flat[selected_indices] = True
    return noisy, selected_mask

noise_density = 0.30
noise_seed = 42
noisy_gray_30, noise_mask_30 = add_salt_pepper(clean_gray, noise_density, noise_seed)
selected_count_30 = int(np.count_nonzero(noise_mask_30))
pepper_count_30 = int(np.count_nonzero(noise_mask_30 & (noisy_gray_30 == 0)))
salt_count_30 = int(np.count_nonzero(noise_mask_30 & (noisy_gray_30 == 255)))
changed_count_30 = int(np.count_nonzero(noisy_gray_30 != clean_gray))

assert selected_count_30 == round(noise_density * clean_gray.size)
assert pepper_count_30 + salt_count_30 == selected_count_30
assert np.array_equal(noisy_gray_30[~noise_mask_30], clean_gray[~noise_mask_30])
assert np.array_equal(clean_gray, gray), "The original reference was modified."
print(f"NumPy version: {np.__version__}; generator: default_rng; seed: {noise_seed}")
print(f"Noise model: salt-and-pepper replacement; requested density: {noise_density:.0%}")
print("Location sampling: uniform, without replacement; balanced salt/pepper assignments")
print(f"Total spatial pixels: {clean_gray.size:,}")
print(f"Selected pixels: {selected_count_30:,} ({selected_count_30 / clean_gray.size:.4%})")
print(f"Pepper assignments (0): {pepper_count_30:,} ({pepper_count_30 / clean_gray.size:.4%} of image)")
print(f"Salt assignments (255): {salt_count_30:,} ({salt_count_30 / clean_gray.size:.4%} of image)")
print(f"Actually changed values: {changed_count_30:,} ({changed_count_30 / clean_gray.size:.4%})")
print(f"Selected but already equal to assigned value: {selected_count_30 - changed_count_30:,}")
print(f"Unselected pixels: {clean_gray.size - selected_count_30:,}; all unchanged: True")
print(f"Noisy image shape: {noisy_gray_30.shape}; dtype: {noisy_gray_30.dtype}")
print(f"Observed noisy intensity range: [{noisy_gray_30.min()}, {noisy_gray_30.max()}]")
print(f"Reference preserved: {np.array_equal(clean_gray, gray)}")

NumPy version: 2.3.5; generator: default_rng; seed: 42
Noise model: salt-and-pepper replacement; requested density: 30%
Location sampling: uniform, without replacement; balanced salt/pepper assignments
Total spatial pixels: 307,020
Selected pixels: 92,106 (30.0000%)
Pepper assignments (0): 46,053 (15.0000% of image)
Salt assignments (255): 46,053 (15.0000% of image)
Actually changed values: 91,717 (29.8733%)
Selected but already equal to assigned value: 389
Unselected pixels: 214,914; all unchanged: True
Noisy image shape: (476, 645); dtype: uint8
Observed noisy intensity range: [0, 255]
Reference preserved: True


**Interpretation:** Exactly 30% of locations are selected, with 15% of the image assigned pepper and 15% assigned salt. The printed changed-value count may be lower for the reason above. Retain `noisy_gray_30` as the common input for all three filters so comparisons use the same noise realization. The next step displays this image.

**References — official documentation:**
- [NumPy: Generator.choice](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.choice.html) — uniform sampling and sampling without replacement.
- [NumPy: Random Generator](https://numpy.org/doc/stable/reference/random/generator.html) — `default_rng` and seed initialization.